# Notebook 11: Date & Time Operations
**Filename:** `11_Date_Time.ipynb`  
**Topics Covered:** Datetime, Timestamp, Date Parsing, Timedelta, Time Zones, Extracting Date Features

---

## 1. Timestamps & Datetime Objects (`pd.Timestamp`, `pd.to_datetime`)

### Concept Explanation
Pandas represents single point-in-time moments using `pd.Timestamp` (built on Python's `datetime.datetime`). `pd.to_datetime()` converts strings, numbers, or sequences into Pandas `datetime64[ns]` objects.

### Real-world Example
Converting text strings like `'2026-08-20'` from CSV logs into queryable datetime objects.

### Business Example
Standardizing purchase time entries across transactions recorded across different global offices.

### AI/ML Example
Converting raw time strings into continuous datetime indices for time-series forecasting models.

In [1]:
import pandas as pd

# Creating single Timestamps
ts = pd.Timestamp('2026-08-20 14:30:00')

# Sample raw date strings
raw_dates = pd.Series(['2026-01-15', '2026-02-20', '2026-03-25'])

# Convert Series to datetime64[ns]
dt_series = pd.to_datetime(raw_dates)

print("Single Timestamp:", ts)
print("\nConverted Datetime Series:\n", dt_series)

Single Timestamp: 2026-08-20 14:30:00

Converted Datetime Series:
 0   2026-01-15
1   2026-02-20
2   2026-03-25
dtype: datetime64[us]


---

## 2. Date Parsing & Handling Errors (`format`, `errors`)

### Concept Explanation
`pd.to_datetime()` handles varied date formats using the `format` parameter (e.g., `'%d/%m/%Y'`). Passing `errors='coerce'` converts unparseable or corrupted date strings into `NaT` (Not a Time) missing values without breaking pipeline execution.

### Real-world Example
Parsing log files containing inconsistent mixed format strings (e.g., `'20/08/2026'` vs corrupted entry `'invalid_date'`).

### Business Example
Safely ingesting customer birthdate inputs submitted through web forms where users entered invalid text strings.

### AI/ML Example
Cleaning noisy timestamp features in uncurated web scraping datasets.

In [2]:
import pandas as pd

messy_dates = pd.Series(['20/08/2026', '15/09/2026', 'corrupted_text', '31/12/2026'])

# Parse custom format (%d/%m/%Y) and coerce errors to NaT
parsed_dates = pd.to_datetime(messy_dates, format='%d/%m/%Y', errors='coerce')

print("Parsed Datetime with Coerced Errors:\n", parsed_dates)

Parsed Datetime with Coerced Errors:
 0   2026-08-20
1   2026-09-15
2          NaT
3   2026-12-31
dtype: datetime64[us]


---

## 3. Time Durations & Differences (`pd.Timedelta`)

### Concept Explanation
`pd.Timedelta` represents a duration or difference between two datetime instances. Subtracting two datetime objects yields a `Timedelta` Series, allowing operations like adding offset intervals (`pd.Timedelta(days=7)`).

### Real-world Example
Calculating duration intervals between sensor reboot events.

### Business Example
Computing shipping turnaround times by subtracting `order_date` from `delivery_date` to monitor fulfillment SLAs.

### AI/ML Example
Creating time-elapsed features like customer tenure in days since account creation.

In [3]:
import pandas as pd

df_orders = pd.DataFrame({
    'Order_Date': pd.to_datetime(['2026-08-01', '2026-08-05']),
    'Delivery_Date': pd.to_datetime(['2026-08-04', '2026-08-10'])
})

# Calculate duration (Timedelta)
df_orders['Delivery_Time_Days'] = (df_orders['Delivery_Date'] - df_orders['Order_Date']).dt.days

# Add fixed offset duration (7 days)
df_orders['Return_Deadline'] = df_orders['Delivery_Date'] + pd.Timedelta(days=7)

print("Orders with Delivery Durations and Deadlines:\n", df_orders)

Orders with Delivery Durations and Deadlines:
   Order_Date Delivery_Date  Delivery_Time_Days Return_Deadline
0 2026-08-01    2026-08-04                   3      2026-08-11
1 2026-08-05    2026-08-10                   5      2026-08-17


---

## 4. Time Zone Handling (`tz_localize`, `tz_convert`)

### Concept Explanation
By default, timestamps are time-zone naive. `tz_localize()` attaches a initial time zone to naive timestamps, while `tz_convert()` translates localized timestamps from one time zone to another (e.g., UTC to US/Eastern or Asia/Kolkata).

### Real-world Example
Converting UTC flight departure timestamps into local airport time zones.

### Business Example
Normalizing order timestamp records gathered from international e-commerce servers to UTC for corporate auditing.

### AI/ML Example
Standardizing global transaction timestamps to UTC before generating hourly feature aggregations.

In [4]:
import pandas as pd

# Create time-zone naive datetime
naive_dt = pd.Series(pd.date_range('2026-08-20 12:00', periods=2, freq='H'))

# Localize to UTC
utc_dt = naive_dt.dt.tz_localize('UTC')

# Convert from UTC to IST (Asia/Kolkata)
ist_dt = utc_dt.dt.tz_convert('Asia/Kolkata')

print("UTC Timestamps:\n", utc_dt)
print("\nConverted to Asia/Kolkata:\n", ist_dt)

ValueError: Invalid frequency: H. Failed to parse with error message: ValueError("Invalid frequency: H. Failed to parse with error message: KeyError('H'). Did you mean h?") Did you mean h?

---

## 5. Extracting Date Features (`.dt` Accessor)

### Concept Explanation
The `.dt` accessor grants access to vectorized datetime properties (`year`, `month`, `day`, `dayofweek`, `day_name()`, `quarter`, `is_weekend`) from `datetime64` Series.

### Real-world Example
Extracting month numbers from long-term temperature logs to analyze annual climate seasonality.

### Business Example
Extracting day names and weekend flags from store transactions to analyze weekday vs. weekend sales performance.

### AI/ML Example
Engineering temporal features (e.g., `hour`, `dayofweek`, `is_month_end`) for machine learning models.

In [5]:
import pandas as pd

df_events = pd.DataFrame({
    'Event_Time': pd.date_range('2026-08-18', periods=4, freq='D')
})

# Extract temporal features using .dt
df_events['Year'] = df_events['Event_Time'].dt.year
df_events['Month'] = df_events['Event_Time'].dt.month
df_events['Day_Name'] = df_events['Event_Time'].dt.day_name()
df_events['Is_Weekend'] = df_events['Event_Time'].dt.dayofweek >= 5

print("Extracted Temporal Features:\n", df_events)

Extracted Temporal Features:
   Event_Time  Year  Month   Day_Name  Is_Weekend
0 2026-08-18  2026      8    Tuesday       False
1 2026-08-19  2026      8  Wednesday       False
2 2026-08-20  2026      8   Thursday       False
3 2026-08-21  2026      8     Friday       False


---

## Minimum 5 Interview Questions with Answers

1. **What is the difference between a time-zone naive and a time-zone aware timestamp in Pandas?**  
   * **Answer:** A time-zone naive timestamp contains no explicit time zone information (assumed to be local or wall-clock time). A time-zone aware timestamp explicitly references a UTC offset or IANA time zone identifier (e.g., `UTC` or `Asia/Kolkata`).

2. **How does `errors='coerce'` behave in `pd.to_datetime()`?**  
   * **Answer:** When invalid or unparseable date strings are encountered, `errors='coerce'` converts those specific bad values into `NaT` (Not a Time) rather than raising a Python exception and halting the script.

3. **What is the purpose of the `.dt` accessor in Pandas?**  
   * **Answer:** The `.dt` accessor exposes vectorized datetime properties and methods (e.g., `.dt.year`, `.dt.day_name()`, `.dt.days`) on Series containing `datetime64` or `timedelta64` objects.

4. **How do you calculate the exact number of days between two datetime columns?**  
   * **Answer:** Subtract the two datetime columns to produce a `Timedelta` Series, and then call `.dt.days` to extract the integer count of days (e.g., `(df['End'] - df['Start']).dt.days`).

5. **What is the difference between `tz_localize()` and `tz_convert()`?**  
   * **Answer:** `tz_localize()` assigns a time zone to a naive datetime that has no time zone attached. `tz_convert()` translates an already localized datetime from one time zone to another.

---

## Self Reflection
* **What I Learned:** Mastered Timestamp creation (`to_datetime`), error handling (`errors='coerce'`), duration arithmetic (`Timedelta`), time zone management (`tz_localize`/`tz_convert`), and temporal feature extraction (`.dt` accessor).
* **Key Takeaway:** Correct date parsing and temporal feature engineering are foundational for analyzing time-series datasets and generating time-based machine learning features.